# KIS baseline — seeds 42 and 1234

Three-seed study together with the existing seed 2345. Each new run uses 300 total epochs, the same teacher/data/recipe, and its matching seed-specific KIS warmup. Evaluation runs only at epoch 300; shared warmup performs no evaluation.

Default: `INPUT_MODE = 'upload'`. Upload `iad_cba_v2_code.zip`, `btcv_data.zip`, and `epoch_299.pth` into `/content/` using Colab's Files panel. Wait for every upload to finish before setup. SAM base weights download automatically.

For Drive input, set `INPUT_MODE = 'drive'`; place the code/data ZIPs under `MyDrive/IAD` and the teacher under `MyDrive/IAD/teacher_weights/epoch_299.pth`.

Run cells 1–5 first to produce and download both warmups. Upload warmup_s42.pth and warmup_s1234.pth into their corresponding Ours runtimes. Then run the KIS training cell for both seeds sequentially.

Upload-only runtimes save locally. Download results before the runtime ends; local files cannot provide recovery after Colab discards the VM. A mounted Drive folder enables checkpoint backup every five epochs.

In [ ]:
# 1. CONFIG
INPUT_MODE = "upload"  # "upload" for /content files; "drive" to mount Google Drive.
RUN = "kis"
SEEDS = [42, 1234]
EPOCHS = 300
WORK = "/content/IAD"
RUNS = "/content/runs/main"
DRIVE_ROOT = "/content/drive/MyDrive/IAD"

In [ ]:
# 2. SETUP: use uploaded files in /content, or switch INPUT_MODE to drive.
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

if INPUT_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    INPUT = Path(DRIVE_ROOT)
    MAIN_DRIVE = str(INPUT / "main")
elif INPUT_MODE == "upload":
    INPUT = Path("/content")
    MAIN_DRIVE = str(Path(DRIVE_ROOT) / "main") if os.path.isdir(DRIVE_ROOT) else None
else:
    raise ValueError("INPUT_MODE must be 'upload' or 'drive'")

work = Path(WORK)
work.mkdir(parents=True, exist_ok=True)
for name in ("iad_cba_v2_code.zip", "btcv_data.zip"):
    archive = INPUT / name
    if not archive.is_file() or not zipfile.is_zipfile(archive):
        raise RuntimeError(f"Missing or incomplete ZIP: {archive}. Finish uploading the file before running setup.")
    with zipfile.ZipFile(archive) as z:
        bad = z.testzip()
        if bad:
            raise RuntimeError(f"Corrupt ZIP entry: {bad} in {archive}")
        z.extractall(work)

tea_src = INPUT / "epoch_299.pth"
if not tea_src.is_file():
    tea_src = INPUT / "teacher_weights" / "epoch_299.pth"
if not tea_src.is_file():
    raise FileNotFoundError(f"Upload epoch_299.pth into {INPUT}")
tea_dst = work / "teacher_weights" / "epoch_299.pth"
tea_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(tea_src, tea_dst)
sam_dst = work / "data" / "weights" / "sam_vit_l_0b3195.pth"
sam_dst.parent.mkdir(parents=True, exist_ok=True)
sam_src = INPUT / "data" / "weights" / sam_dst.name
if not sam_dst.is_file() or sam_dst.stat().st_size < 1_200_000_000:
    if sam_src.is_file():
        shutil.copy2(sam_src, sam_dst)
    else:
        subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(sam_dst),
                        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth"], check=True)
assert sam_dst.stat().st_size >= 1_200_000_000, "Incomplete SAM base weights"
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
if f"{WORK}/scripts" not in sys.path:
    sys.path.insert(0, f"{WORK}/scripts")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements_colab.txt"], check=True)

# Freeze the original v5 parameters for both new seeds; do not read a changing tuning state.
STATE = str(work / "multiseed_v5_state.json")
Path(STATE).write_text(json.dumps({"lambda_a": 0.02, "tau": 0.1, "m": 0.2, "r": 2, "beta": 0.5}))

def run_command(command):
    print("Running:", " ".join(command), flush=True)
    process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f"Training command failed with exit code {code}")

def command_for(seed, dry_run=False):
    cmd = [sys.executable, f"scripts/run_main_{RUN}.py", "--runs_dir", RUNS,
           "--epochs", str(EPOCHS), "--seed", str(seed), "--eval_freq", "0"]
    if MAIN_DRIVE:
        cmd += ["--drive_dir", MAIN_DRIVE]
    if RUN == "ours":
        cmd += ["--arm", "v5", "--state_json", STATE, "--no-make_warmup", "--wait_min", "1440"]
    else:
        cmd += ["--make_warmup"]
    if dry_run:
        cmd.append("--dry_run")
    return cmd
print("Input mode:", INPUT_MODE, "| checkpoint backup:", MAIN_DRIVE or "local /content only")

In [ ]:
# 3. VERIFY: environment, bundled launcher, data, and seed-isolated dry runs.
import importlib, torch
assert torch.cuda.is_available(), "Select a GPU runtime in Colab first"
print("Torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
for module in ("icecream", "cv2", "einops", "medpy", "h5py", "timm", "scipy"):
    importlib.import_module(module)
import datasets.dataset_synapse as ds
assert Path(ds.__file__).resolve().is_relative_to(Path(WORK).resolve()), ds.__file__
assert (work / "data/lists_Synapse/train.txt").is_file()
assert (work / "data/btcv/test_vol_h5").is_dir()
import main_common as mc
assert mc.seed_name("warmup", 42) == "warmup_s42", "Old code ZIP: upload the rebuilt bundle"
for seed in SEEDS:
    run_command(command_for(seed, dry_run=True))

In [ ]:
# 4. KIS: create both paired warmups before starting either 300-epoch main run.
# Each warmup uses the full 300-epoch learning-rate schedule and finishes at epoch 20.
for seed in SEEDS:
    checkpoint = mc.find_warmup_ckpt(RUNS, MAIN_DRIVE, seed)
    if checkpoint is None:
        run_command(mc.warmup_command(EPOCHS, RUNS, MAIN_DRIVE, seed))
        checkpoint = mc.find_warmup_ckpt(RUNS, MAIN_DRIVE, seed)
    if checkpoint is None:
        raise RuntimeError(f"Warmup missing for seed {seed}")
    exported = Path(f"/content/warmup_s{seed}.pth")
    shutil.copy2(checkpoint, exported)
    print(f"Paired warmup ready: {exported}", flush=True)

In [ ]:
# 5. Upload mode: download the warmups, then upload each into its Ours runtime /content/.
# Drive mode: the Ours notebooks read the shared checkpoints directly.
if INPUT_MODE == "upload":
    from google.colab import files
    for seed in SEEDS:
        files.download(f"/content/warmup_s{seed}.pth")
else:
    print("Warmups available on Drive; no manual transfer required.")

In [ ]:
# RUN: continue from epoch 20 to epoch 300 (total budget: 300, not 320).
# Re-running resumes only this seed's latest checkpoint or skips its completed run.
for seed in SEEDS:
    run_command(command_for(seed))

In [ ]:
# STATUS: report each seed from its local arm folder, or the Drive mirror.
import csv, glob
for seed in SEEDS:
    name = mc.seed_name("kis300" if RUN == "kis" else "ours_v5", seed)
    arm = "Arm_V1_KIS" if RUN == "kis" else "Arm_V5_Dual_CEHard"
    local = Path(RUNS) / name / arm
    mirror = Path(MAIN_DRIVE) / name if MAIN_DRIVE else local
    directory = local if (local / "metrics.csv").exists() else mirror
    metric_path = directory / "metrics.csv"
    if metric_path.is_file():
        with metric_path.open() as handle:
            rows = list(csv.DictReader(handle))
        if rows:
            print(name, "latest epoch:", rows[-1].get("epoch"), "| seconds:", rows[-1].get("time_sec"))
    final = directory / "eval_final.json"
    if final.is_file():
        overall = json.loads(final.read_text())["overall"]
        print(name, {key: value for key, value in overall.items() if key.startswith("primary_")})

In [ ]:
# DOWNLOAD: save results/checkpoints before ending an upload-only runtime.
from google.colab import files
for seed in SEEDS:
    name = mc.seed_name("kis300" if RUN == "kis" else "ours_v5", seed)
    directory = Path(RUNS) / name
    assert directory.is_dir(), f"No results found at {directory}"
    archive = shutil.make_archive(f"/content/results_{name}", "zip", root_dir=RUNS, base_dir=name)
    files.download(archive)